# Thornfield — Training Pipeline

Full pipeline: `validate → pack → train Hopfield → train policy → benchmark`

---

## How to run (pick one)

### A — Google Drive  ✓ recommended
Files persist between sessions. Outputs stay in your Drive.
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Run the **Mount Drive** cell below — it will ask to authorise once
3. First time only: upload a zip of your repo to Drive (see instructions in the cell)
4. Run all remaining cells in order

### B — GitHub clone
No file upload needed, but requires the repo to be on GitHub (private is fine).
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Set `GITHUB_URL` and optionally `GITHUB_TOKEN` in the **Config** cell
3. Run all cells in order

### C — Local upload to Colab session
Files are lost when the session ends. Good for a quick one-off run.
1. Runtime → Change runtime type → **T4 GPU** → Save
2. Use the Colab file browser (left sidebar → folder icon) to upload a zip
3. Run the unzip cell, then all remaining cells

---

| Stage | Output |
|---|---|
| 00 Setup | paths resolved, deps installed |
| 01 Validate | case JSON verified |
| 02 Pack | `cases/amber_cipher/` |
| 03 Train Hopfield | `outputs/amber_cipher/model.pt` |
| 04 Train Policy | `outputs/amber_cipher/policy.pt` |
| 05 Benchmark | Xcode recommendation |
| 06 Download | `.pt` files to your machine |

---
## 00 — Setup

In [ ]:
import subprocess, sys, os

# ── GPU check ──────────────────────────────────────────────────────────────
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                    '--format=csv,noheader'], capture_output=True, text=True)
if r.returncode == 0:
    print(f'GPU : {r.stdout.strip()}')
    DEVICE = 'cuda'
else:
    print('No GPU — using CPU (slower)')
    DEVICE = 'cpu'

print(f'Device: {DEVICE}')

In [ ]:
import os, sys, subprocess

# ═══════════════════════════════════════════════════════════════════════════
#  CHOOSE YOUR METHOD — uncomment exactly ONE block below
# ═══════════════════════════════════════════════════════════════════════════

# ── A: Google Drive ─────────────────────────────────────────────────────────
#
# First-time setup (do this once from your local machine):
#   1. Zip the repo:   zip -r more_than_words.zip more_than_words/
#   2. Upload the zip to your Google Drive (any folder is fine)
#
# Then set DRIVE_ZIP_PATH to where you put it, e.g.:
#   '/content/drive/MyDrive/more_than_words.zip'         (Drive root)
#   '/content/drive/MyDrive/projects/more_than_words.zip'
#
# If you already extracted it before, set DRIVE_ZIP_PATH = None and
# set DRIVE_REPO_DIR to the extracted folder path instead.

METHOD         = 'drive'
DRIVE_ZIP_PATH = '/content/drive/MyDrive/more_than_words.zip'  # ← update this
DRIVE_REPO_DIR = '/content/drive/MyDrive/more_than_words'      # extracted location

# ── B: GitHub clone ─────────────────────────────────────────────────────────
# For private repos, create a Personal Access Token at github.com/settings/tokens
# Leave GITHUB_TOKEN = '' for public repos.

# METHOD       = 'github'
# GITHUB_URL   = 'https://github.com/YOUR_ORG/more_than_words.git'
# GITHUB_TOKEN = ''   # paste token here for private repos, or leave empty

# ── C: Zip already uploaded to session storage ──────────────────────────────
# Use the Colab file browser (left sidebar) to upload more_than_words.zip first.

# METHOD   = 'session'
# ZIP_PATH = '/content/more_than_words.zip'   # wherever you uploaded it

# ═══════════════════════════════════════════════════════════════════════════
#  Resolution logic — do not edit below this line
# ═══════════════════════════════════════════════════════════════════════════

REPO_DIR = None

if METHOD == 'drive':
    from google.colab import drive
    print('Mounting Google Drive...')
    drive.mount('/content/drive')

    if DRIVE_ZIP_PATH and os.path.exists(DRIVE_ZIP_PATH):
        if not os.path.exists(DRIVE_REPO_DIR):
            print(f'Extracting {DRIVE_ZIP_PATH} → {os.path.dirname(DRIVE_REPO_DIR)}')
            os.makedirs(os.path.dirname(DRIVE_REPO_DIR), exist_ok=True)
            subprocess.run(['unzip', '-q', DRIVE_ZIP_PATH,
                            '-d', os.path.dirname(DRIVE_REPO_DIR)], check=True)
        else:
            print(f'Already extracted at {DRIVE_REPO_DIR}')
        REPO_DIR = DRIVE_REPO_DIR
    elif os.path.isdir(DRIVE_REPO_DIR):
        print(f'Using existing folder: {DRIVE_REPO_DIR}')
        REPO_DIR = DRIVE_REPO_DIR
    else:
        raise FileNotFoundError(
            f'Nothing found at {DRIVE_ZIP_PATH} or {DRIVE_REPO_DIR}.\n'
            'Upload more_than_words.zip to Google Drive and update DRIVE_ZIP_PATH.'
        )

elif METHOD == 'github':
    REPO_DIR = '/content/more_than_words'
    if not os.path.exists(REPO_DIR):
        url = GITHUB_URL
        if GITHUB_TOKEN:
            # embed token for private repos
            url = url.replace('https://', f'https://{GITHUB_TOKEN}@')
        print(f'Cloning...')
        subprocess.run(['git', 'clone', url, REPO_DIR], check=True)
    else:
        print('Repo already present.')
        subprocess.run(['git', '-C', REPO_DIR, 'pull'])

elif METHOD == 'session':
    REPO_DIR = '/content/more_than_words'
    if not os.path.exists(REPO_DIR):
        print(f'Extracting {ZIP_PATH}...')
        subprocess.run(['unzip', '-q', ZIP_PATH, '-d', '/content'], check=True)
    else:
        print('Already extracted.')

else:
    raise ValueError(f'Unknown METHOD: {METHOD}')

# ── Common: set paths and Python sys.path ───────────────────────────────────
TRAINER_DIR = os.path.join(REPO_DIR, 'thornfield', 'trainer')
assert os.path.isdir(TRAINER_DIR), f'trainer not found at {TRAINER_DIR} — check REPO_DIR'

if TRAINER_DIR not in sys.path:
    sys.path.insert(0, TRAINER_DIR)
os.chdir(TRAINER_DIR)

print(f'\nMethod      : {METHOD}')
print(f'REPO_DIR    : {REPO_DIR}')
print(f'TRAINER_DIR : {TRAINER_DIR}')
print(f'cwd         : {os.getcwd()}')

In [ ]:
# ── Quick sanity check ────────────────────────────────────────────────────
# Run this if the setup cell raised an error to see what's actually at /content/

print('=== /content/ ===')
!ls /content/

if METHOD == 'drive':
    print('\n=== Google Drive root ===')
    !ls /content/drive/MyDrive/ | head -20

print('\n=== trainer/ ===')
!ls {TRAINER_DIR}

In [ ]:
# ── Install dependencies ───────────────────────────────────────────────────
!pip install -q -r {TRAINER_DIR}/requirements.txt
print('Dependencies ready.')

In [ ]:
# ── Pipeline configuration — edit here ────────────────────────────────────
CASE_ID = 'amber_cipher'   # 'amber_cipher' (3-dim, 72 tokens)
                           # 'amber_cipher_M' (5-dim, 152 tokens — see bottom)

# Stage 03 — Hopfield model
HOPFIELD_PATHS    = 300    # 300 = fast (~2 min GPU), 500 = stronger signal
HOPFIELD_EPOCHS   = 20
PROOF_PATHS       = 200
PROOF_ATTEMPTS    = 2000

# Stage 04 — Transformer policy (strict KD + REINFORCE)
SUPERVISED_PATHS  = 2000   # Stage 1 demonstrations
SUPERVISED_EPOCHS = 20
RL_EPISODES       = 500    # Stage 2 rollouts
KD_TEMPERATURE    = 0.5    # teacher softmax sharpness
                           # amber_cipher: T=0.5 gives P(inv)~8%, T=0.35 gives ~20%
                           # T=2.0 is too flat for weights in [0,1]
KD_ALPHA          = 0.3    # 0 = all soft KD, 1 = all hard CE
KD_COEF           = 0.05   # RL anchor weight

# Stage 05 — Benchmark
BENCHMARK_EPISODES = 200

# ── Subprocess env (suppresses threading conflicts on Colab / macOS) ───────
PROC_ENV = {
    **os.environ,
    'PYTHONPATH': TRAINER_DIR,
    'PYTHONUNBUFFERED': '1',
    'KMP_DUPLICATE_LIB_OK': 'TRUE',
    'OMP_NUM_THREADS': '1',
    'MKL_NUM_THREADS': '1',
}

print(f'Case   : {CASE_ID}')
print(f'Device : {DEVICE}')
print(f'Mode   : {MODE}  ({"outputs sync back to VS Code" if MODE=="vscode" else "use cell 06 to download"})')

---
## 01 — Validate case JSON

Checks token counts, class distribution, invariant purity, attractor gradients, graph symmetry.

In [ ]:
case_json = os.path.join(REPO_DIR, f'{CASE_ID}.json')
validator = os.path.join(REPO_DIR, 'thornfield_case_validator.py')

r = subprocess.run(['python3', validator, case_json], capture_output=True, text=True)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr)
    raise RuntimeError(f'Validation failed for {CASE_ID}')
print('✓ Case JSON valid')

---
## 02 — Pack case

`amber_cipher.json` → `cases/amber_cipher/{spec,tokens,graph,attractor,...}.json`

In [ ]:
packer = os.path.join(TRAINER_DIR, 'tools', 'pack_case.py')
r = subprocess.run(['python3', packer, case_json], capture_output=True, text=True,
                   env=PROC_ENV, cwd=TRAINER_DIR)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr)
    raise RuntimeError('Pack failed')

import glob
packed = sorted(glob.glob(f'cases/{CASE_ID}/*.json'))
print(f'Packed files ({len(packed)}):')
for f in packed:
    print(f'  {f}')

In [ ]:
from core.cartridge import CartridgeSpec

spec = CartridgeSpec.load(f'cases/{CASE_ID}/spec.json')

print(f'Case            : {spec.case_id}')
print(f'Vocab size      : {spec.vocab_size}')
print(f'Attractor dims  : {spec.n_attractor_dims}')
print(f'Convergence rate: {spec.convergence_rate}')
print(f'Turn window     : {spec.min_turns} – {spec.max_turns}')
print(f'Threshold       : {spec.convergence_threshold}')
print()
print('Invariant tokens:')
for inv_id in spec.invariant_token_ids:
    tok = spec.get_token(inv_id)
    print(f'  {inv_id:35s}  weights={tok.attractor_weights}')

---
## 03 — Train Hopfield model

Trains `MysteryEnergyModel` → `outputs/amber_cipher/model.pt`  
Then runs proof gate: Lyapunov + basin coverage + convergence rate + invariant accuracy.

In [ ]:
cmd = [
    'python3', 'tools/train_single_case.py', CASE_ID,
    '--paths',              str(HOPFIELD_PATHS),
    '--epochs',             str(HOPFIELD_EPOCHS),
    '--proof-paths',        str(PROOF_PATHS),
    '--proof-max-attempts', str(PROOF_ATTEMPTS),
    '--device',             DEVICE,
]
print('$', ' '.join(cmd), '\n')

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, env=PROC_ENV, cwd=TRAINER_DIR)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError('Hopfield training failed')

In [ ]:
model_path = f'outputs/{CASE_ID}/model.pt'
assert os.path.exists(model_path), f'model.pt not found — did training complete?'
print(f'model.pt  {os.path.getsize(model_path)/1e6:.2f} MB  ✓')
if MODE == 'vscode':
    print(f'  syncing to local: {os.path.join(REPO_DIR, "thornfield", "trainer", model_path)}')

In [ ]:
import json, matplotlib.pyplot as plt

history_path = f'outputs/{CASE_ID}/history.json'
if not os.path.exists(history_path):
    print('history.json not found — skipping plot'); raise SystemExit

with open(history_path) as f:
    h = json.load(f)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for key, label, ls in [
    ('total_loss',      'total',         '-'),
    ('retrieval_loss',  'retrieval',     '--'),
    ('energy_margin',   'energy margin', ':'),
]:
    if key in h:
        ax1.plot(h[key], label=label, linestyle=ls)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Hopfield training loss'); ax1.legend(); ax1.grid(alpha=0.3)

proof_keys = ['proof_lyapunov', 'proof_basin', 'proof_convergence']
if all(k in h for k in proof_keys):
    ax2.bar(['Lyapunov', 'Basin', 'Convergence'],
            [h['proof_lyapunov'], h['proof_basin'], h['proof_convergence']],
            color=['steelblue', 'coral', 'mediumseagreen'])
    ax2.axhline(0.9, color='red', linestyle='--', label='pass bar 90%')
    ax2.set_ylim(0, 1.05); ax2.legend()
else:
    ax2.text(0.5, 0.5, 'proof metrics\nnot in history.json',
             ha='center', va='center', transform=ax2.transAxes, color='gray')
ax2.set_title('Proof gate')

plt.tight_layout()
plt.savefig(f'outputs/{CASE_ID}/hopfield_training.png', dpi=120)
plt.show()

---
## 04 — Train transformer policy

**Stage 1 — Strict KD** from Hopfield attractor weights (not a model — the weight matrix directly):  
```
soft_targets[d] = softmax(attractor_weights[:, d] / T)
loss = α · CE(logits, hard_label)  +  (1-α) · T² · KL(softmax(logits/T) ∥ soft_targets)
```

**Stage 2 — REINFORCE** with energy reward + KD anchor:  
```
loss = -Σ log_π · G  −  0.01 · entropy  +  kd_coef · KL(policy ∥ soft_targets)
```

In [ ]:
cmd = [
    'python3', 'trainer/train_policy.py', CASE_ID,
    '--supervised-paths',  str(SUPERVISED_PATHS),
    '--supervised-epochs', str(SUPERVISED_EPOCHS),
    '--rl-episodes',       str(RL_EPISODES),
    '--kd-temperature',    str(KD_TEMPERATURE),
    '--kd-alpha',          str(KD_ALPHA),
    '--kd-coef',           str(KD_COEF),
    '--device',            DEVICE,
]
print('$', ' '.join(cmd), '\n')

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, env=PROC_ENV, cwd=TRAINER_DIR)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError('Policy training failed')

In [ ]:
import torch

policy_path = f'outputs/{CASE_ID}/policy.pt'
assert os.path.exists(policy_path), 'policy.pt not found'
print(f'policy.pt  {os.path.getsize(policy_path)/1e6:.2f} MB  ✓')

ckpt = torch.load(policy_path, map_location='cpu')
print(f'case_id      : {ckpt.get("case_id")}')
print(f'model_type   : {ckpt.get("model_type")}')
print(f'n_dims       : {ckpt.get("n_attractor_dims")}')
print(f'parameters   : {sum(v.numel() for v in ckpt["state_dict"].values()):,}')
if MODE == 'vscode':
    print(f'  syncing to local: {os.path.join(REPO_DIR, "thornfield", "trainer", policy_path)}')

---
## 05 — Benchmark: Hopfield vs Transformer

200 rollouts each. Full proof gate on transformer.  
Ends with the **XCODE MODEL RECOMMENDATION** block.

In [ ]:
cmd = [
    'python3', 'tools/benchmark_models.py', CASE_ID,
    '--n-episodes', str(BENCHMARK_EPISODES),
    '--plot',
]
print('$', ' '.join(cmd), '\n')

proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, env=PROC_ENV, cwd=TRAINER_DIR)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError('Benchmark failed')

In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg

plot_path = f'outputs/{CASE_ID}/benchmark_turns.png'
if os.path.exists(plot_path):
    img = mpimg.imread(plot_path)
    plt.figure(figsize=(10, 4))
    plt.imshow(img); plt.axis('off')
    plt.title('Turn distribution — Hopfield vs Transformer')
    plt.tight_layout(); plt.show()

---
## 06 — Download outputs

> **VS Code mode**: skip this. Files sync back to your local machine automatically.  
> **Browser mode**: run this cell to download `.pt` files.

In [ ]:
from google.colab import files

print('Downloading outputs...')
for name, path in [
    ('Hopfield model',      f'outputs/{CASE_ID}/model.pt'),
    ('Transformer policy',  f'outputs/{CASE_ID}/policy.pt'),
]:
    full = os.path.join(TRAINER_DIR, path)
    if os.path.exists(full):
        size_mb = os.path.getsize(full) / 1e6
        print(f'  {name}: {size_mb:.2f} MB')
        files.download(full)
    else:
        print(f'  {name}: NOT FOUND at {full}')

if METHOD == 'drive':
    print('\nNote: files are also in your Google Drive at:')
    print(f'  {os.path.join(TRAINER_DIR, f"outputs/{CASE_ID}/")}'  )

---
## Extras — Debugging & Inspection

### Attractor weight distribution

In [ ]:
import numpy as np, matplotlib.pyplot as plt

weights    = np.array([t.attractor_weights for t in spec.tokens])  # (V, n_dims)
n_dims     = spec.n_attractor_dims
dim_labels = ['suspect', 'mechanism', 'motive', 'location', 'accomplice'][:n_dims]

fig, axes = plt.subplots(1, n_dims, figsize=(5 * n_dims, 4))
if n_dims == 1: axes = [axes]

for d, ax in enumerate(axes):
    w = weights[:, d]
    ax.hist(w[w < 0.9], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    inv_id  = spec.invariant_token_ids[d]
    inv_idx = next(i for i, t in enumerate(spec.tokens) if t.id == inv_id)
    ax.axvline(weights[inv_idx, d], color='crimson', lw=2, label=f'invariant ({inv_id})')
    ax.set_title(f'dim {d}: {dim_labels[d]}')
    ax.set_xlabel('attractor weight'); ax.legend(fontsize=8)

plt.suptitle(f'{CASE_ID} — attractor weight distribution', y=1.02)
plt.tight_layout(); plt.show()

### KD teacher distribution (what the policy is asked to match)

In [ ]:
import torch, torch.nn.functional as F

T = KD_TEMPERATURE
all_w = torch.tensor(np.array([t.attractor_weights for t in spec.tokens]), dtype=torch.float32)
soft  = F.softmax(all_w.T / T, dim=-1)  # (n_dims, V)

fig, axes = plt.subplots(1, n_dims, figsize=(5 * n_dims, 4))
if n_dims == 1: axes = [axes]

for d, ax in enumerate(axes):
    probs = soft[d].numpy()
    top10 = np.argsort(probs)[::-1][:10]
    labels = [spec.tokens[i].id.replace('_', ' ') for i in top10]
    colors = ['crimson' if spec.tokens[top10[j]].is_invariant else 'steelblue'
              for j in range(10)]
    ax.barh(range(10), probs[top10][::-1], color=colors[::-1])
    ax.set_yticks(range(10)); ax.set_yticklabels(labels[::-1], fontsize=8)
    ax.set_xlabel('teacher probability')
    ax.set_title(f'dim {d}: {dim_labels[d]}')
    H = -(soft[d] * soft[d].clamp(1e-12).log()).sum().item()
    ax.text(0.97, 0.03, f'H={H:.2f}', transform=ax.transAxes,
            ha='right', va='bottom', fontsize=9, color='gray')

plt.suptitle(f'KD teacher distribution  T={T}  (crimson = invariant)', y=1.02)
plt.tight_layout(); plt.show()

### Sample 5 Hopfield paths

In [ ]:
from generator.path_sampler import PathSampler

sampler = PathSampler(spec, sampling_temperature=1.4, min_affinity=0.05, allow_partial=False)
paths   = sampler.sample_batch(5, verbose=False)
print(f'Sampled {len(paths)} converging paths\n')

for i, path in enumerate(paths):
    print(f'Path {i+1}  ({len(path)} turns):')
    for turn, triad in enumerate(path):
        ids    = [t.id for t in triad]
        marker = ' ← SOLUTION' if all(t.is_invariant for t in triad) else ''
        print(f'  turn {turn+1:2d}  {ids}{marker}')
    print()

### Greedy policy rollout

In [ ]:
from trainer.energy_model import MysteryEnergyModel
from rl.casebook_env import CasebookEnv
from core.token import TokenClass, TokenPhase, TokenStream, TokenAgency

ckpt  = torch.load(f'outputs/{CASE_ID}/policy.pt', map_location='cpu')
model = MysteryEnergyModel(
    vocab_size=spec.vocab_size, embedding_dim=spec.embedding_dim,
    context_dim=spec.context_dim, n_attractor_dims=spec.n_attractor_dims,
    token_graph=spec.token_graph,
)
model.load_state_dict(ckpt['state_dict'])
model.eval()

id2i  = {t.id: i for i, t in enumerate(spec.tokens)}
cls2i = {c.value: i for i, c in enumerate(TokenClass)}
pha2i = {p.value: i for i, p in enumerate(TokenPhase)}
str2i = {s.value: i for i, s in enumerate(TokenStream)}
age2i = {a.value: i for i, a in enumerate(TokenAgency)}

env  = CasebookEnv(spec, spec.token_graph)
obs  = env.reset()
done = False
print('=== Greedy rollout ===')
print(f'Opening: {obs["placed_token_ids"]}')

while not done:
    toks = [spec.get_token(tid) for tid in obs['placed_token_ids']]
    pos  = [(min(i // 3, 7), i % 3) for i in range(len(toks))]

    with torch.no_grad():
        emb = model.token_embedding(
            torch.tensor([id2i[t.id]             for t in toks], dtype=torch.long),
            torch.tensor([cls2i[t.token_class.value] for t in toks], dtype=torch.long),
            torch.tensor([pha2i[t.phase.value]    for t in toks], dtype=torch.long),
            torch.tensor([str2i[t.stream.value]   for t in toks], dtype=torch.long),
            torch.tensor([age2i[t.agency.value]   for t in toks], dtype=torch.long),
        )
        ctx    = model.casebook_encoder(
            emb.unsqueeze(0),
            torch.tensor(pos, dtype=torch.float32).unsqueeze(0),
            torch.zeros(1, len(toks), dtype=torch.bool),
        )
        all_emb = model.token_embedding.token_emb(torch.arange(spec.vocab_size))
        logits  = model.retrieval_head(ctx, all_emb)  # (1, n_dims, V)

    placed_set = set(obs['placed_token_ids'])
    action = []
    for d in range(spec.n_attractor_dims):
        dl = logits[0, d, :].clone()
        for tid in placed_set:
            dl[id2i[tid]] = float('-inf')
        action.append(spec.tokens[dl.argmax().item()].id)

    obs, reward, done, info = env.step(action)
    print(f'  turn {info["turn"]:2d}  {action}  reward={reward:+.3f}  '
          f'conv={info.get("convergence_score", 0):.3f}')

print()
print(f'Converged : {info["converged"]}  |  Turns : {info["turn"]}  |  '
      f'Correct : {info["correct_invariants"]}')

---
## amber_cipher_M — 5-dim (152 tokens)

Uncomment the cell below, then re-run cells 01–06.

In [ ]:
# CASE_ID           = 'amber_cipher_M'
# HOPFIELD_PATHS    = 500
# SUPERVISED_PATHS  = 2000
# RL_EPISODES       = 500
# BENCHMARK_EPISODES = 200
# print('Switched to', CASE_ID)